# AIkenGPT MMLU-style Evaluation — Letter scoring

このNotebookはOpenAI評価と同じ `MMLUEvaluator` を使います。
通常は次の **User settings** だけを変更し、GPU runtimeでRun allしてください。

処理順: Settings → Environment setup → Model loading → Dataset / evaluator setup
→ Preflight / manifest → Evaluation → Results


## 1. Settings

### User settings — 通常はこのセルだけを変更

In [ ]:
from pathlib import Path

# ========================================
# User settings
# ========================================

DATA_DIR = Path("/content/drive/MyDrive/datasets/mmlu")
OUTPUT_DIR = Path("/content/drive/MyDrive/aikengpt_mmlu_results")

SUBJECT = None          # 例: "abstract_algebra"。Noneは全subject
NTRAIN = 5              # 0ならzero-shot
SAMPLE_FRAC = 0.10
SEED = 42
LIMIT = 0               # 0ならsampled questionsをすべて評価

# Noneならこのrun用に生成。同じ問題集合を再利用するときは既存manifestを指定。
MANIFEST_PATH = None


### Project settings — 通常の問題セット変更では編集不要

In [ ]:
# Repository / fixed AIkenGPT settings
PROJECT_DIR = Path("/content/drive/MyDrive/AIkenSGTv1_New")
MODEL_REPO = "HayatoHongo/AIkenSGTv1"
MODEL_REVISION = None   # 再現性を厳密にする場合はimmutable revisionを指定
MODEL_PATH = None       # local .safetensorsを使う場合だけPathを指定
MODEL_ID = "HayatoHongo/AIkenSGTv1:model_clean.safetensors"
TOKENIZER = "gpt2"

DEVICE = "cuda"
DTYPE = "float32"
BATCH_SIZE = 1
MAX_CONTEXT_LENGTH = 2048
CONTEXT_POLICY = "reduce"  # 共通2048-token budget内まで両modelで同じようにshot数を減らす
CONTEXT_TOKENIZER = "gpt2"
PERMUTATION_COUNT = 4
PERMUTATION_SEED = 0       # cyclic orderは固定。記録用で乱数には未使用

SCORING_METHOD = "letter"
OUTPUT_NAME = f"aikengpt_{NTRAIN}shot_letter.csv"


## 2. Environment setup

In [ ]:
%pip install -q numpy pandas tiktoken safetensors huggingface_hub


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, str(PROJECT_DIR))

assert (PROJECT_DIR / "mmlu_eval").is_dir(), (
    "PROJECT_DIR must point to the repository containing mmlu_eval/"
)
assert DATA_DIR.is_dir(), "DATA_DIR must contain dev/ and test/"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Model loading

In [ ]:
import torch
import tiktoken
from huggingface_hub import hf_hub_download
from mmlu_eval.backends.aikengpt_backend import (
    AIkenGPTBackend,
    file_sha256,
    load_custom_checkpoint,
)

assert DEVICE != "cuda" or torch.cuda.is_available(), "Select a Colab GPU runtime"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

if MODEL_PATH is None:
    MODEL_PATH = Path(hf_hub_download(
        repo_id=MODEL_REPO,
        filename="model_clean.safetensors",
        revision=MODEL_REVISION,
    ))
else:
    MODEL_PATH = Path(MODEL_PATH)

checkpoint_hash = file_sha256(MODEL_PATH)
tokenizer = tiktoken.get_encoding(TOKENIZER)
model = load_custom_checkpoint(
    MODEL_PATH,
    device=DEVICE,
    dtype=DTYPE,
    max_context_length=MAX_CONTEXT_LENGTH,
)
print("Checkpoint SHA256:", checkpoint_hash)


## 4. Dataset / evaluator setup

In [ ]:
from mmlu_eval import EvalConfig, MMLUEvaluator

eval_config = EvalConfig(
    ntrain=NTRAIN,
    sample_frac=SAMPLE_FRAC,
    seed=SEED,
    limit=LIMIT,
    subject=SUBJECT,
    permutation_count=PERMUTATION_COUNT,
    permutation_seed=PERMUTATION_SEED,
    context_policy=CONTEXT_POLICY,
    context_tokenizer=CONTEXT_TOKENIZER,
    max_context_length=MAX_CONTEXT_LENGTH,
)
evaluator = MMLUEvaluator(DATA_DIR, eval_config)


## 5. Preflight / manifest

In [ ]:
from mmlu_eval.core import atomic_csv

# Existing manifest is authoritative. Otherwise create one once and reuse it below.
manifest = evaluator.manifest(MANIFEST_PATH)
preflight_manifest_path = OUTPUT_DIR / (Path(OUTPUT_NAME).stem + ".preflight.manifest.csv")
preflight_prompts_path = OUTPUT_DIR / (Path(OUTPUT_NAME).stem + ".preflight.prompts.jsonl")
atomic_csv(preflight_manifest_path, manifest)
evaluator.export_debug(manifest, preflight_prompts_path)
active_manifest_path = Path(MANIFEST_PATH) if MANIFEST_PATH is not None else preflight_manifest_path

first = manifest.iloc[0]
first_case = evaluator.cases(first.subject, int(first.test_index))[0]
print(f"Questions: {len(manifest)} / prompts: {len(manifest) * 4}")
print("Manifest:", active_manifest_path)
print("\nFirst prompt:\n")
print(first_case.prompt)


## 6. Evaluation

In [ ]:
backend = AIkenGPTBackend(
    model,
    tokenizer,
    model_identifier=MODEL_ID,
    tokenizer_identifier=TOKENIZER,
    scoring_method=SCORING_METHOD,
    text_reduction=None,
    device=DEVICE,
    dtype=DTYPE,
    batch_size=BATCH_SIZE,
    max_context_length=MAX_CONTEXT_LENGTH,
    checkpoint_sha256=checkpoint_hash,
)

OUTPUT_PATH = OUTPUT_DIR / OUTPUT_NAME
results = evaluator.run(
    backend,
    OUTPUT_PATH,
    manifest_path=active_manifest_path,
)


## 7. Results

In [ ]:
from mmlu_eval.core import summarize

print(summarize(results))
print("Result CSV:", OUTPUT_PATH)
print("Manifest:", OUTPUT_PATH.with_suffix(".manifest.csv"))
print("Prompts:", OUTPUT_PATH.with_suffix(".prompts.jsonl"))
display(results.head())

# OpenAI runとの入力一致を確認する場合:
# from mmlu_eval.compare import compare
# compare("/path/to/openai.prompts.jsonl", OUTPUT_PATH.with_suffix(".prompts.jsonl"))
